In [1]:
import oracledb
import pandas as pd
from datetime import datetime

In [11]:
# Database connection parameters
dsn = oracledb.makedsn("10.5.224.170", 1521, service_name="xepdb1")
username = "bank_reviews"
password = "password123"

In [12]:
# Sample data (replace with CSV loading or parse provided text)
clean_reviews = [
    {"review": "the app is proactive and a good connections.", "rating": 5, "date": "2025-06-05", "bank": "CBE", "source": "Google Play"},
    {"review": "I cannot send to cbebirr app. through this app.", "rating": 3, "date": "2025-06-05", "bank": "CBE", "source": "Google Play"},
    # Add more rows from clean_bank_reviews
]
analysis_results = [
    {"review_id": 0, "sentiment_label": "positive", "sentiment_score": 0.9998316764831543, "themes": "['Other']"},
    {"review_id": 1, "sentiment_label": "negative", "sentiment_score": 0.9793662428855896, "themes": "['Other']"},
    # Add more rows from analysis_results
]

In [13]:
# Convert to DataFrames
clean_df = pd.DataFrame(clean_reviews)
analysis_df = pd.DataFrame(analysis_results)

# Merge datasets on review text (assuming review text is unique for matching)
merged_df = pd.merge(clean_df, analysis_df, left_index=True, right_index=True, how='left')

# Clean themes (remove list notation)
merged_df['themes'] = merged_df['themes'].str.replace(r"[\[\]']", "", regex=True)

In [22]:
# Connect to Oracle
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    cursor = connection.cursor()

    # Get bank_id for CBE
    # Check if the bank exists
    cursor.execute(
        "SELECT bank_id FROM bank_reviews.Banks WHERE bank_name = :1",
        ("Commercial Bank of Ethiopia",)
    )
    result = cursor.fetchone()

    if result is None:
        # Insert the bank
        cursor.execute(
            "INSERT INTO bank_reviews.Banks (bank_name, app_name) VALUES (:1, :2)",
            ("Commercial Bank of Ethiopia", "CBE Mobile Banking")
        )
        connection.commit()  # commit the new bank
        print("Inserted missing bank.")

        # Re-fetch the bank_id
        cursor.execute(
            "SELECT bank_id FROM bank_reviews.Banks WHERE bank_name = :1",
            ("Commercial Bank of Ethiopia",)
        )
        result = cursor.fetchone()

    # Proceed if bank_id is now available
    if result:
        bank_id = result[0]
        print("Bank ID:", bank_id)

        # Insert reviews as before
        insert_query = """
        INSERT INTO bank_reviews.Reviews (bank_id, review_text, rating, review_date, source, sentiment_label, sentiment_score, themes)
        VALUES (:1, :2, :3, :4, :5, :6, :7, :8)
        """
        for _, row in merged_df.iterrows():
            cursor.execute(insert_query, (
                bank_id,
                row['review'],
                row['rating'],
                datetime.strptime(row['date'], '%Y-%m-%d'),
                row['source'],
                row['sentiment_label'],
                row['sentiment_score'],
                row['themes']
            ))

        connection.commit()
        print(f"Inserted {len(merged_df)} reviews successfully.")
    else:
        print("Failed to retrieve or insert bank.")

        
    banks_to_insert = [
        ("Bank of Abyssinia", "BOA Mobile Banking"),
        ("Dashen Bank", "Dashen Mobile Banking")
    ]

    for bank_name, app_name in banks_to_insert:
        # Check if bank already exists
        cursor.execute(
            "SELECT bank_id FROM bank_reviews.Banks WHERE bank_name = :1",
            (bank_name,)
        )
        if cursor.fetchone() is None:
            cursor.execute(
                "INSERT INTO bank_reviews.Banks (bank_name, app_name) VALUES (:1, :2)",
                (bank_name, app_name)
            )
            print(f"Inserted bank: {bank_name}")

    connection.commit()


except oracledb.Error as e:
    print(f"Error: {e}")
finally:
    cursor.close()
    connection.close()

# Export SQL dump (schema and data)
# Use Oracle Data Pump or SQL Developer to export the database as an SQL file

Bank ID: 4
Inserted 2 reviews successfully.
Inserted bank: Bank of Abyssinia
Inserted bank: Dashen Bank


In [23]:
# Connect to Oracle
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    cursor = connection.cursor()

    # Get bank_id for CBE
    cursor.execute("SELECT bank_id FROM bank_reviews.Banks WHERE bank_name = :1", ("Commercial Bank of Ethiopia",))
    bank_id = cursor.fetchone()[0]

    # Insert reviews
    insert_query = """
    INSERT INTO bank_reviews.Reviews (bank_id, review_text, rating, review_date, source, sentiment_label, sentiment_score, themes)
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8)
    """
    for _, row in merged_df.iterrows():
        cursor.execute(insert_query, (
            bank_id,
            row['review'],
            row['rating'],
            datetime.strptime(row['date'], '%Y-%m-%d'),
            row['source'],
            row['sentiment_label'],
            row['sentiment_score'],
            row['themes']
        ))

    # Commit the transaction
    connection.commit()
    print(f"Inserted {len(merged_df)} reviews successfully.")

except oracledb.Error as e:
    print(f"Error: {e}")
finally:
    cursor.close()
    connection.close()

Inserted 2 reviews successfully.


In [28]:
import oracledb

dsn = "10.5.224.170:1521/xepdb1"
username = "bank_reviews"
password = "password123"

connection = oracledb.connect(user=username, password=password, dsn=dsn)
cursor = connection.cursor()

def export_table_data(table_name):
    cursor.execute(f"SELECT * FROM {table_name}")
    columns = [desc[0] for desc in cursor.description]

    with open(f"{table_name.replace('.', '_').lower()}_data.sql", "w") as f:
        for row in cursor:
            values = []
            for val in row:
                if val is None:
                    values.append("NULL")
                elif isinstance(val, str):
                    safe_val = val.replace("'", "''")
                    values.append(f"'{safe_val}'")
                elif isinstance(val, (int, float)):
                    values.append(str(val))
                elif hasattr(val, 'isoformat'):  # dates, timestamps
                    values.append(f"TO_DATE('{val.strftime('%Y-%m-%d %H:%M:%S')}', 'YYYY-MM-DD HH24:MI:SS')")
                else:
                    values.append(f"'{str(val)}'")
            insert_stmt = f"INSERT INTO {table_name} ({', '.join(columns)}) VALUES ({', '.join(values)});\n"
            f.write(insert_stmt)

export_table_data('bank_reviews.BANKS')
export_table_data('bank_reviews.REVIEWS')

cursor.close()
connection.close()
